# RAG 是什么？
RAG = Retrieval-Augmented Generation（检索增强生成）

简单说就是：让AI在回答问题之前，先去查资料，然后再回答。

# 文本分割：
## 文本分割主要考虑两个因素：
1）embedding模型的Tokens限制情况；2）语义完整性对整体的检索效果的影响。一些常见的文本分割方式如下：

句分割：以”句”的粒度进行切分，保留一个句子的完整语义。常见切分符包括：句号、感叹号、问号、换行符等。
固定长度分割：根据embedding模型的token长度限制，将文本分割为固定长度（如1024/512个tokens），这种切分方式会损失很多语义信息，一般通过在头尾增加一定冗余量来缓解。

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 加载文档
loader = TextLoader("crossover_epic_saga.txt", encoding="utf-8")
documents = loader.load()

# 2. 切分文档
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每段500字
    chunk_overlap=50,    # 重叠50字，保持连贯
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print(f"原文档: {len(documents)} 个")
print(f"切分后: {len(chunks)} 段")
print(f"\n第一段内容:\n{chunks[0].page_content}")
print(f"\n第二段内容:\n{chunks[1].page_content}")

原文档: 1 个
切分后: 27 段

第一段内容:
# 次元裂缝：洛洛的终极冒险

## 第一章：异变降临

"机车战士们，准备出击！"

洛洛站在时光之城的高塔上，望着远处黑压压的猛兽族大军，握紧了拳头。自从来到机战王的世界，他已经带领机车族经历了无数次战斗，但今天的敌人似乎有些不同。

天空突然裂开了一道巨大的缝隙，紫色的闪电在裂缝中游走。那不是普通的闪电，而是某种更加诡异的力量。

"洛洛！那是什么？"霹雳火仰头望着天空，红色的装甲在闪电的映照下显得格外醒目。

洛洛还没来得及回答，裂缝中突然射出一道金光，将他和霹雳火同时笼罩。在失去意识前的最后一刻，洛洛听到了一个机械般的声音：

"检测到符合条件的'王'之资质，启动次元召唤程序……"

---

当洛洛再次睁开眼睛时，他发现自己躺在一片陌生的沙滩上。碧蓝的大海一望无际，远处有几艘帆船正在航行。最让他震惊的是，霹雳火就躺在他身边，但体型变小了——不再是那台巨大的机车战士，而是变成了只有一人高的机器人形态。

"霹雳火！你没事吧？"

"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

第二段内容:
"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传送到了另一个世界，"洛洛分析道，"而且这里的规则不同，你们的体型和力量都受到了限制。"

就在这时，沙滩另一头传来一阵喧闹声。洛洛和霹雳火警惕地望去，只见一个戴着草帽的少年正被一群海盗模样的人追赶。

"把财宝交出来，小子！"

"我才不要呢！这是我找到的！"草帽少年灵活地躲避着攻击，脸上挂着灿烂的笑容，"我可是要成为海贼王的男人！"

洛洛瞳孔一缩。海贼王？他当然知道这个著名的漫画世界。但为什么他会来到这里？那道裂缝到底是什么？

"需要帮忙吗？"洛洛走上前问道。

草帽少年——路飞愣了一下，然后大笑起来："哈哈哈！不用不用，这种小角色我自己就能搞定！橡胶橡胶——手枪！"

他的手臂突然伸长，一拳将追在最前面的海盗打飞了出去。

洛洛和霹雳火对视一眼，都从对方眼中看到了震惊。这个世界的能力体系……完全不同于机战王的世界！

---


# 向量化（embedding）：
向量化是一个将文本数据转化为向量矩阵的过程，该过程会直接影响到后续检索的效果。目前常见的embedding模型基本能满足大部分需求，但对于特殊场景（例如涉及一些罕见专有词或字等）或者想进一步优化效果，则可以选择开源Embedding模型微调或直接训练适合自己场景的Embedding模型。这里嵌入模型使用Qwen3-Embedding-0.6B

In [1]:
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import numpy as np

# 模型路径
model_path = r"C:\Users\吴创捷\Desktop\生产\agent开发\qwen3-embedding-model\Qwen3-Embedding-0.6B\qwen\Qwen3-Embedding-0.6B"

print("🚀 加载模型中...")
print(f"路径: {model_path}")

# 用 SentenceTransformer 加载（Transformer → Pooling → Normalize 流水线）
# trust_remote_code 去掉，模型无 auto_map 不需要
model = SentenceTransformer(model_path)

print("✅ 模型加载成功！")
print(f"模型类型: {type(model).__name__}")

# 测试文本
texts = [
    "洛洛是机战王",
    "路飞是要成为海贼王的男人",
    "孙悟空会龟派气功",
    "今天天气真好"
]

print(f"\n📝 测试 {len(texts)} 个句子:\n")

# 批量编码
embeddings = model.encode(texts, normalize_embeddings=True)

for text, emb in zip(texts, embeddings):
    print(f"文本: {text}")
    print(f"向量维度: {emb.shape}")
    print(f"向量前5个值: {emb[:5]}")
    print()

# 测试语义相似度
print("🔍 测试语义相似度:")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

text1 = "洛洛是机战王"
text2 = "洛洛操控机车战士"
text3 = "今天吃了苹果"

emb1 = model.encode(text1, normalize_embeddings=True)
emb2 = model.encode(text2, normalize_embeddings=True)
emb3 = model.encode(text3, normalize_embeddings=True)

print(f"'{text1}' vs '{text2}': {cosine_similarity(emb1, emb2):.4f}")
print(f"'{text1}' vs '{text3}': {cosine_similarity(emb1, emb3):.4f}")
print(f"\n相似度越高表示语义越接近！")

🚀 加载模型中...
路径: C:\Users\吴创捷\Desktop\生产\agent开发\qwen3-embedding-model\Qwen3-Embedding-0.6B\qwen\Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ 模型加载成功！
模型类型: SentenceTransformer

📝 测试 4 个句子:

文本: 洛洛是机战王
向量维度: (1024,)
向量前5个值: [ 0.05932617 -0.01318359 -0.00567627 -0.04858398  0.00634766]

文本: 路飞是要成为海贼王的男人
向量维度: (1024,)
向量前5个值: [ 0.08203125  0.02832031 -0.00860596 -0.03881836  0.01098633]

文本: 孙悟空会龟派气功
向量维度: (1024,)
向量前5个值: [-0.0062561  -0.05126953 -0.00848389 -0.0078125   0.0559082 ]

文本: 今天天气真好
向量维度: (1024,)
向量前5个值: [-0.02062988 -0.01464844 -0.00817871 -0.0112915   0.05859375]

🔍 测试语义相似度:
'洛洛是机战王' vs '洛洛操控机车战士': 0.7296
'洛洛是机战王' vs '今天吃了苹果': 0.2351

相似度越高表示语义越接近！
